# 03 · Data Warehouse: modelo en estrella (R3 · Hito 2)
**Proyecto semestral · Sala de crisis COVID-19 · Gestión de Datos IN1232C · 2026-2**

Requiere `01_ingesta` y `02_calidad`. Construye la **capa analítica** en DuckDB:

```
                 dim_fecha                      dim_territorio
              (fecha_key PK)                 (territorio_key PK)
                     \                       /   jerarquía:
                      \                     /    país → región/estado → condado
                    fact_covid_diario                + población, lat/long, ISO3
          grano: 1 fila = 1 territorio × 1 día
          medidas: acumulados, nuevos diarios, banderas de calidad, linaje
```
La integración con la fuente externa (OWID vacunación y Banco Mundial) se agrega en la siguiente iteración como dimensión/hechos adicionales por ISO3.

## 1 · ¿Por qué DuckDB? (justificación del motor)
| Criterio | DuckDB (elegido) | SQLite | PostgreSQL | MongoDB (NoSQL, S4) |
|---|---|---|---|---|
| Tipo de carga | **Analítica (OLAP)**: agregaciones sobre millones de filas | Transaccional (OLTP) | OLTP y OLAP | Documentos, OLTP |
| Almacenamiento | **Columnar**, vectorizado | Por filas | Por filas | Documentos JSON (BSON) |
| Instalación | `pip install duckdb`, un archivo; corre en Colab | Un archivo | Servidor | Servidor o Atlas |
| Lee Parquet directo con SQL | **Sí** | No | Con extensiones | No |
| Modelo en estrella / JOIN / OLAP (`ROLLUP`, `CUBE`) | **Sí** | Sin `ROLLUP` | Sí | `$lookup` limitado; sin esquema fijo |

Nuestros datos son **tabulares, con esquema conocido y de solo lectura** (el repositorio está archivado), y las preguntas son agregaciones por tiempo y territorio. Un motor columnar embebido es la opción natural. MongoDB conviene cuando el esquema cambia por registro o hay muchas escrituras concurrentes de documentos; aquí el esquema cambió solo 5 veces y ya está unificado, así que su flexibilidad no aporta y sus agregaciones con JOIN serían más lentas y verbosas.

## 2 · Configuración

In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install -q duckdb

In [2]:
import time
from pathlib import Path
import pandas as pd
import duckdb

EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROYECTO = Path('/content/drive/MyDrive/proyecto-gestion-datos-2026')
else:
    PROYECTO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CLEAN = PROYECTO / 'data' / 'clean'
DW = PROYECTO / 'data' / 'dw'; DW.mkdir(parents=True, exist_ok=True)
assert (CLEAN / 'homologacion_paises.parquet').exists(), 'Primero ejecuta 01_ingesta y 02_calidad'
pd.set_option('display.max_columns', None)

diario = pd.read_parquet(CLEAN / 'daily_reports.parquet')
uid = pd.read_parquet(CLEAN / 'uid_lookup.parquet')
homol = pd.read_parquet(CLEAN / 'homologacion_paises.parquet')
print(f'{len(diario):,} filas en la capa limpia')

4,287,473 filas en la capa limpia


## 3 · Preparación (transformaciones del ETL)
1. Homologar país (tabla de 02) y pasar territorios de ultramar a provincia.
2. Si la provincia repite el nombre del país (`France|France`, mar-2020) se trata como fila de nivel país.
3. **Deduplicar** la llave territorio × día: se conserva la última fila del archivo (decisión de 02).

In [3]:
d = diario.merge(homol[['country_region', 'pais', 'es_territorio_ultramar']], on='country_region', how='left')
d['province_state'] = d['province_state'].astype('object')
d['admin2'] = d['admin2'].astype('object')
m = d['es_territorio_ultramar'] & d['province_state'].isna()
d.loc[m, 'province_state'] = d.loc[m, 'country_region'].astype(str)
d.loc[d['province_state'] == d['pais'].astype(str), 'province_state'] = None

txt = lambda s: s.where(s.notna(), '').astype(str)
d['clave_territorio'] = d['pais'].astype(str) + '|' + txt(d['province_state']) + '|' + txt(d['admin2'])

antes = len(d)
d = d.sort_values(['fecha_reporte', 'archivo_origen', 'fila_origen']).drop_duplicates(['fecha_reporte', 'clave_territorio'], keep='last')
print(f'Duplicados eliminados: {antes - len(d)} filas')

Duplicados eliminados: 78 filas


## 4 · Dimensión territorio
Se parte de la **tabla UID** (4.321 territorios con población y coordenadas) y se agregan los territorios que aparecen en los reportes pero no en la tabla (por ejemplo, ciudades de EE.UU. informadas como «provincia» en feb-2020, o *Repatriated Travellers* de Canadá). Estos quedan con `en_tabla_uid = False` y sin población.

In [4]:
u = uid.copy()
u['clave_territorio'] = u['country_region'] + '|' + txt(u['province_state']) + '|' + txt(u['admin2'])
u = u.rename(columns={'country_region': 'pais', 'province_state': 'region', 'admin2': 'condado', 'population': 'poblacion',
                      'iso3': 'iso3_territorio'})
u['en_tabla_uid'] = True

faltan = d.loc[~d['clave_territorio'].isin(u['clave_territorio'])].groupby('clave_territorio').agg(
    pais=('pais', 'first'), region=('province_state', 'first'), condado=('admin2', 'first'),
    lat=('lat', 'median'), long=('long', 'median')).reset_index()
faltan['en_tabla_uid'] = False

dim_territorio = pd.concat([u, faltan], ignore_index=True)
dim_territorio['nivel'] = 'pais'
dim_territorio.loc[dim_territorio['region'].notna(), 'nivel'] = 'region'
dim_territorio.loc[dim_territorio['condado'].notna(), 'nivel'] = 'condado'
iso_pais = homol.drop_duplicates('pais').set_index('pais')['iso3']
dim_territorio['iso3_pais'] = dim_territorio['pais'].map(iso_pais)
dim_territorio['entidad_especial'] = dim_territorio['pais'].isin(homol.loc[homol['entidad_especial'], 'pais'])
# coordenadas (0,0) no son válidas para mapas (regla V2)
cero = (dim_territorio['lat'] == 0) & (dim_territorio['long'] == 0)
dim_territorio.loc[cero, ['lat', 'long']] = None
dim_territorio.insert(0, 'territorio_key', range(1, len(dim_territorio) + 1))
dim_territorio = dim_territorio[['territorio_key', 'uid', 'clave_territorio', 'nivel', 'pais', 'region', 'condado',
                                 'iso3_pais', 'iso3_territorio', 'fips', 'lat', 'long', 'poblacion',
                                 'entidad_especial', 'en_tabla_uid']]

print(dim_territorio['nivel'].value_counts().to_dict(), '| fuera de la tabla UID:', (~dim_territorio['en_tabla_uid']).sum())
cobertura = d['clave_territorio'].isin(u['clave_territorio']).mean() * 100
print(f'Filas de hechos que cruzan con la tabla UID: {cobertura:.2f} %')
dim_territorio[dim_territorio['pais'] == 'Chile']

{'condado': 3363, 'region': 936, 'pais': 202} | fuera de la tabla UID: 180
Filas de hechos que cruzan con la tabla UID: 99.93 %


,territorio_key,uid,clave_territorio,nivel,pais,region,condado,iso3_pais,iso3_territorio,fips,lat,long,poblacion,entidad_especial,en_tabla_uid
74,75,152.0,Chile||,pais,Chile,None,None,CHL,CHL,<NA>,-35.6751,-71.5430,19116209,False,True
75,76,15201.0,Chile|Antofagasta|,region,Chile,Antofagasta,None,CHL,CHL,<NA>,-23.6509,-70.3975,607534,False,True
76,77,15202.0,Chile|Araucania|,region,Chile,Araucania,None,CHL,CHL,<NA>,-38.9489,-72.3311,957224,False,True
77,78,15203.0,Chile|Arica y Parinacota|,region,Chile,Arica y Parinacota,None,CHL,CHL,<NA>,-18.5940,-69.4785,226068,False,True
78,79,15204.0,Chile|Atacama|,region,Chile,Atacama,None,CHL,CHL,<NA>,-27.5661,-70.0503,288944,False,True
79,80,15205.0,Chile|Aysen|,region,Chile,Aysen,None,CHL,CHL,<NA>,-45.9864,-73.7669,103158,False,True
80,81,15206.0,Chile|Biobio|,region,Chile,Biobio,None,CHL,CHL,<NA>,-37.4464,-72.1416,1556805,False,True
81,82,15207.0,Chile|Coquimbo|,region,Chile,Coquimbo,None,CHL,CHL,<NA>,-29.9590,-71.3389,757586,False,True
82,83,15208.0,Chile|Los Lagos|,region,Chile,Los Lagos,None,CHL,CHL,<NA>,-41.9198,-72.1416,828708,False,True
83,84,15209.0,Chile|Los Rios|,region,Chile,Los Rios,None,CHL,CHL,<NA>,-40.2310,-72.3311,384837,False,True


## 5 · Dimensión fecha

In [5]:
fechas = pd.date_range('2020-01-22', '2023-03-09', freq='D')
MESES = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']
dim_fecha = pd.DataFrame({
    'fecha_key': fechas.strftime('%Y%m%d').astype(int),
    'fecha': fechas,
    'anio': fechas.year, 'trimestre': fechas.quarter, 'mes': fechas.month,
    'nombre_mes': [MESES[m - 1] for m in fechas.month],
    'anio_mes': fechas.strftime('%Y-%m'),
    'semana_iso': fechas.isocalendar().week.values, 'anio_iso': fechas.isocalendar().year.values,
    'dia_semana': fechas.dayofweek + 1,      # 1 = lunes
    'es_fin_de_semana': fechas.dayofweek >= 5,
})
dim_fecha.head()

,fecha_key,fecha,anio,trimestre,mes,nombre_mes,anio_mes,semana_iso,anio_iso,dia_semana,es_fin_de_semana
0,20200122,2020-01-22,2020,1,1,enero,2020-01,4,2020,3,False
1,20200123,2020-01-23,2020,1,1,enero,2020-01,4,2020,4,False
2,20200124,2020-01-24,2020,1,1,enero,2020-01,4,2020,5,False
3,20200125,2020-01-25,2020,1,1,enero,2020-01,4,2020,6,True
4,20200126,2020-01-26,2020,1,1,enero,2020-01,4,2020,7,True


## 6 · Tabla de hechos
* **Grano:** un territorio en un día de reporte.
* **Medidas:** acumulados (`confirmados_acum`, `fallecidos_acum`) y nuevos diarios (`casos_nuevos`, `fallecidos_nuevos`) = diferencia con el reporte anterior del **mismo territorio**.
* El primer día de un territorio los nuevos quedan **nulos** (no se asume que todo el acumulado es nuevo). Así, cuando Chile pasa a regiones (20-may-2020), no aparece un «pico» falso: los nuevos nacionales se calculan agregando acumulados (roll-up), no sumando diferencias de territorios que recién aparecen.
* **Banderas de calidad (R2):** `flag_acum_baja`, `flag_archivo_defectuoso` (reporte del 23-may-2022).
* **Linaje:** `archivo_origen`, `fila_origen`.

In [6]:
ARCHIVOS_DEFECTUOSOS = {'05-23-2022.csv'}     # hallazgo de 02_calidad

f = d.merge(dim_territorio[['territorio_key', 'clave_territorio']], on='clave_territorio', how='left', validate='many_to_one')
assert f['territorio_key'].notna().all()
f = f.sort_values(['territorio_key', 'fecha_reporte'])
f['fecha_key'] = f['fecha_reporte'].dt.strftime('%Y%m%d').astype(int)
g = f.groupby('territorio_key')
f['casos_nuevos'] = g['confirmed'].diff()
f['fallecidos_nuevos'] = g['deaths'].diff()
f['flag_acum_baja'] = (f['casos_nuevos'] < 0) | (f['fallecidos_nuevos'] < 0)
f['flag_archivo_defectuoso'] = f['archivo_origen'].astype(str).isin(ARCHIVOS_DEFECTUOSOS)

fact = f.rename(columns={'confirmed': 'confirmados_acum', 'deaths': 'fallecidos_acum', 'recovered': 'recuperados_acum'})[[
    'fecha_key', 'territorio_key', 'confirmados_acum', 'fallecidos_acum', 'recuperados_acum',
    'casos_nuevos', 'fallecidos_nuevos', 'flag_acum_baja', 'flag_archivo_defectuoso',
    'archivo_origen', 'fila_origen']].reset_index(drop=True)
fact['archivo_origen'] = fact['archivo_origen'].astype(str)
print(f'{len(fact):,} filas de hechos')
fact.head()

4,287,395 filas de hechos


,fecha_key,territorio_key,confirmados_acum,fallecidos_acum,recuperados_acum,casos_nuevos,fallecidos_nuevos,flag_acum_baja,flag_archivo_defectuoso,archivo_origen,fila_origen
0,20200224,1,5,0,0,<NA>,<NA>,<NA>,False,02-24-2020.csv,63
1,20200225,1,5,0,0,0,0,False,False,02-25-2020.csv,67
2,20200226,1,5,0,0,0,0,False,False,02-26-2020.csv,75
3,20200227,1,5,0,0,0,0,False,False,02-27-2020.csv,77
4,20200228,1,5,0,0,0,0,False,False,02-28-2020.csv,79


## 7 · Carga en DuckDB

In [7]:
BD = DW / 'covid_dw.duckdb'
if BD.exists(): BD.unlink()
con = duckdb.connect(str(BD))
t0 = time.time()
con.execute('CREATE TABLE dim_fecha AS SELECT * FROM dim_fecha')
con.execute('CREATE TABLE dim_territorio AS SELECT * FROM dim_territorio')
con.execute('CREATE TABLE fact_covid_diario AS SELECT * FROM fact')
con.execute('ALTER TABLE dim_fecha ADD PRIMARY KEY (fecha_key)')
con.execute('ALTER TABLE dim_territorio ADD PRIMARY KEY (territorio_key)')
# Vista de conveniencia: hechos con sus dimensiones
con.execute('''CREATE OR REPLACE VIEW v_hechos AS
    SELECT f.*, d.fecha, d.anio, d.mes, d.anio_mes, d.semana_iso, t.nivel, t.pais, t.region, t.condado,
           t.iso3_pais, t.poblacion, t.lat, t.long, t.entidad_especial
    FROM fact_covid_diario f JOIN dim_fecha d USING (fecha_key) JOIN dim_territorio t USING (territorio_key)''')
print(f'Carga: {time.time() - t0:.1f} s')
con.sql('''SELECT table_name, estimated_size AS filas FROM duckdb_tables() ORDER BY 1''').df()

Carga: 1.5 s


,table_name,filas
0,dim_fecha,1143
1,dim_territorio,4501
2,fact_covid_diario,4287395


In [8]:
# Integridad referencial y grano
print(con.sql('''SELECT
   (SELECT COUNT(*) FROM fact_covid_diario f LEFT JOIN dim_fecha d USING (fecha_key) WHERE d.fecha_key IS NULL) AS hechos_sin_fecha,
   (SELECT COUNT(*) FROM fact_covid_diario f LEFT JOIN dim_territorio t USING (territorio_key) WHERE t.territorio_key IS NULL) AS hechos_sin_territorio,
   (SELECT COUNT(*) - COUNT(DISTINCT (fecha_key, territorio_key)) FROM fact_covid_diario) AS violaciones_de_grano''').df())

   hechos_sin_fecha  hechos_sin_territorio  violaciones_de_grano
0                 0                      0                     0


## 8 · Primeras consultas OLAP sobre el DW
Todas se hacen con SQL sobre el modelo en estrella, no sobre los CSV.

In [9]:
# ROLL-UP: de territorio × día a país × día, y tasa por 100.000 (Chile vs Sudamérica, último día)
SUDAMERICA = ('ARG', 'BOL', 'BRA', 'CHL', 'COL', 'ECU', 'GUY', 'PRY', 'PER', 'SUR', 'URY', 'VEN')
con.sql(f'''
    WITH pais_dia AS (
        SELECT t.iso3_pais, t.pais, f.fecha_key, SUM(f.confirmados_acum) AS confirmados, SUM(f.fallecidos_acum) AS fallecidos
        FROM fact_covid_diario f JOIN dim_territorio t USING (territorio_key)
        GROUP BY ALL),
    pob AS (SELECT iso3_pais, poblacion FROM dim_territorio WHERE nivel = 'pais' AND en_tabla_uid)
    SELECT p.pais, confirmados, fallecidos,
           ROUND(confirmados * 1e5 / pob.poblacion) AS casos_x100k,
           ROUND(fallecidos * 1e5 / pob.poblacion, 1) AS fallecidos_x100k
    FROM pais_dia p JOIN pob USING (iso3_pais)
    WHERE fecha_key = 20230309 AND iso3_pais IN {SUDAMERICA}
    ORDER BY fallecidos_x100k DESC''').df()

,pais,confirmados,fallecidos,casos_x100k,fallecidos_x100k
0,Peru,4487553.0,219539.0,13610.0,665.8
1,Chile,5192286.0,64273.0,27162.0,336.2
2,Brazil,37076053.0,699276.0,17443.0,329.0
3,Argentina,10044957.0,130472.0,22225.0,288.7
4,Colombia,6359093.0,142339.0,12498.0,279.7
5,Paraguay,808401.0,19878.0,11334.0,278.7
6,Suriname,82467.0,1404.0,14058.0,239.3
7,Uruguay,1034303.0,7617.0,29775.0,219.3
8,Ecuador,1057121.0,36014.0,5992.0,204.1
9,Bolivia,1194277.0,22365.0,10231.0,191.6


In [10]:
# DRILL-DOWN: Chile por región (desde que existe el detalle regional) · SLICE: un día
con.sql('''
    SELECT t.region, t.poblacion, f.confirmados_acum, f.fallecidos_acum,
           ROUND(f.confirmados_acum * 1e5 / t.poblacion) AS casos_x100k,
           ROUND(f.fallecidos_acum * 1e5 / t.poblacion, 1) AS fallecidos_x100k
    FROM fact_covid_diario f JOIN dim_territorio t USING (territorio_key)
    WHERE t.pais = 'Chile' AND t.nivel = 'region' AND t.region <> 'Unknown' AND f.fecha_key = 20230309
    ORDER BY fallecidos_x100k DESC''').df()

,region,poblacion,confirmados_acum,fallecidos_acum,casos_x100k,fallecidos_x100k
0,Magallanes,166533,67653,774,40624.0,464.8
1,Metropolitana,7112808,1967389,30869,27660.0,434.0
2,Arica y Parinacota,226068,79718,900,35263.0,398.1
3,Los Rios,384837,148252,1411,38523.0,366.6
4,Valparaiso,1815902,492917,6461,27144.0,355.8
5,Tarapaca,330558,113542,1173,34349.0,354.9
6,Antofagasta,607534,183811,1961,30255.0,322.8
7,Maule,1044950,346804,3354,33189.0,321.0
8,Los Lagos,828708,243060,2577,29330.0,311.0
9,Biobio,1556805,510730,4824,32806.0,309.9


In [11]:
# DICE + ROLL-UP en el tiempo: casos y fallecidos nuevos por año y mes en Biobío vs Metropolitana (2021)
con.sql('''
    SELECT d.anio_mes, t.region, SUM(f.casos_nuevos) AS casos_nuevos, SUM(f.fallecidos_nuevos) AS fallecidos_nuevos
    FROM fact_covid_diario f JOIN dim_fecha d USING (fecha_key) JOIN dim_territorio t USING (territorio_key)
    WHERE t.pais = 'Chile' AND t.region IN ('Biobio', 'Metropolitana') AND d.anio = 2021
    GROUP BY ALL ORDER BY t.region, d.anio_mes''').df().pivot(index='anio_mes', columns='region', values=['casos_nuevos', 'fallecidos_nuevos'])

casos_nuevos               fallecidos_nuevos              
region         Biobio Metropolitana            Biobio Metropolitana
anio_mes                                                           
2021-01       13687.0       24139.0             270.0         560.0
2021-02       11907.0       21079.0             216.0         624.0
2021-03       22485.0       55497.0             276.0         884.0
2021-04       19708.0       77836.0             318.0        1282.0
2021-05       15397.0       78902.0             223.0        1300.0
2021-06       12593.0       76573.0             213.0        1555.0
2021-07        5297.0       20728.0             208.0        1281.0
2021-08        1797.0       10628.0             113.0         659.0
2021-09        1227.0        7647.0              51.0         246.0
2021-10        2056.0       23437.0              20.0         156.0
2021-11        7506.0       26307.0              31.0         331.0
2021-12        6711.0       11474.0             106.0         266.0

In [12]:
# ROLLUP de SQL: subtotales por año y total general de casos nuevos en Chile
con.sql('''
    SELECT COALESCE(CAST(d.anio AS VARCHAR), 'TOTAL') AS anio, SUM(f.casos_nuevos) AS casos_nuevos
    FROM fact_covid_diario f JOIN dim_fecha d USING (fecha_key) JOIN dim_territorio t USING (territorio_key)
    WHERE t.pais = 'Chile'
    GROUP BY ROLLUP (d.anio) ORDER BY anio''').df()

,anio,casos_nuevos
0,2020,604934.0
1,2021,1197521.0
2,2022,3231033.0
3,2023,154759.0
4,TOTAL,5188247.0


> Ojo con la última consulta: la suma de nuevos por territorio **no incluye** el acumulado del primer día de cada región (20-may-2020), por diseño. Para totales nacionales exactos se usa la diferencia de acumulados agregados por país (roll-up primero, diferencia después). Lo formalizaremos como vista `v_pais_dia` en la siguiente iteración.

## 9 · Persistencia

In [13]:
# El DW se guarda también como Parquet por tabla (portátil y liviano para GitHub)
for tabla in ['dim_fecha', 'dim_territorio', 'fact_covid_diario']:
    con.execute(f"COPY {tabla} TO '{DW / (tabla + '.parquet')}' (FORMAT PARQUET, COMPRESSION ZSTD)")
con.close()
for p in sorted(DW.iterdir()):
    print(f'  {p.name:30s} {p.stat().st_size / 1e6:6.1f} MB')

  covid_dw.duckdb                  49.3 MB
  dim_fecha.parquet                 0.0 MB
  dim_territorio.parquet            0.2 MB
  fact_covid_diario.parquet        17.5 MB
